# Text Processing with Python

Text processing usually starts with raw text that needs inspection, cleaning, analysis, and export. Typical tasks include checking line quality, normalizing case and whitespace, removing punctuation or stopwords, counting frequent words, and converting semi-structured text into a table for further analysis.

This notebook follows those steps with six examples adapted from the `scripts/day2` materials. Each section explains the task, defines reusable helpers, and ends with one wrapper function call so the notebook can be run from top to bottom with **Run All**.

## Inspecting a Raw Text File

A text-processing workflow should begin with inspection. Before changing anything, it is useful to see how many lines the file has, which lines are blank, whether digits appear unexpectedly, whether spacing is inconsistent, and whether some answers are duplicated.

In [ ]:
from collections import Counter
from pathlib import Path
import csv
import re
import string


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / '.venv').exists() and (candidate / 'data' / 'day2').exists():
            return candidate
        if (candidate / 'data' / 'day2').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root that contains data/day2.')


PROJECT_ROOT = find_project_root()
DAY2_DATA_DIR = PROJECT_ROOT / 'data' / 'day2'
RAW_PATH = DAY2_DATA_DIR / 'responses_raw.txt'
STOPWORDS_PATH = DAY2_DATA_DIR / 'stopwords_lv.txt'
GENERATED_CLEANED_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses.txt'
GENERATED_FREQ_PATH = DAY2_DATA_DIR / 'generated_word_frequencies.txt'
GENERATED_CLEANED_V2_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses_v2.txt'
GENERATED_FREQ_V2_PATH = DAY2_DATA_DIR / 'generated_word_frequencies_v2.txt'
GENERATED_CLEANED_FINAL_PATH = DAY2_DATA_DIR / 'generated_cleaned_responses_final.txt'
GENERATED_FREQ_FINAL_PATH = DAY2_DATA_DIR / 'generated_word_frequencies_final.txt'
SURVEYS_FOLDER = DAY2_DATA_DIR / 'surveys'
SURVEY_OUTPUT_PATH = SURVEYS_FOLDER / 'survey_responses_summary.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Day 2 data folder: {DAY2_DATA_DIR}')

In [ ]:
def read_text_file(path: Path, encoding: str = 'utf-8') -> str:
    '''Read a text file and return its full contents as one string.'''
    with path.open('r', encoding=encoding) as file:
        return file.read()


def split_into_lines(text: str) -> list[str]:
    '''Split text into lines without keeping trailing newline characters.'''
    return text.splitlines()


def find_blank_lines(lines: list[str]) -> list[int]:
    '''Return 1-based line numbers for blank or whitespace-only lines.'''
    return [i for i, line in enumerate(lines, start=1) if line.strip() == '']


def find_digit_lines(lines: list[str]) -> list[int]:
    '''Return 1-based line numbers for lines containing any digit.'''
    return [i for i, line in enumerate(lines, start=1) if any(ch.isdigit() for ch in line)]


def find_lines_with_extra_spaces(lines: list[str]) -> list[int]:
    '''Return 1-based line numbers for lines containing double spaces.'''
    return [i for i, line in enumerate(lines, start=1) if '  ' in line]


def find_duplicate_lines(lines: list[str]) -> list[tuple[str, int]]:
    '''Return duplicate non-empty lines and how many times they appear.'''
    counts = Counter(line.strip() for line in lines if line.strip() != '')
    return [(line, count) for line, count in counts.items() if count > 1]


def print_inspection_report(lines: list[str]) -> None:
    '''Print a compact inspection report for the raw text.'''
    blank_lines = find_blank_lines(lines)
    digit_lines = find_digit_lines(lines)
    extra_space_lines = find_lines_with_extra_spaces(lines)
    duplicate_lines = find_duplicate_lines(lines)

    print('Raw text inspection')
    print(f'- Total lines: {len(lines)}')
    print(f'- Blank lines: {blank_lines}')
    print(f'- Lines with digits: {digit_lines}')
    print(f'- Lines with repeated spaces: {extra_space_lines}')
    print(f'- Duplicate non-empty lines: {len(duplicate_lines)}')

    print('\nFirst 5 lines')
    for i, line in enumerate(lines[:5], start=1):
        print(f'{i}: {line}')

    if duplicate_lines:
        print('\nDuplicate line examples')
        for line, count in duplicate_lines[:3]:
            print(f'{count}x | {line}')


In [ ]:
def inspect_text_file(path: Path = RAW_PATH) -> dict[str, object]:
    '''Run the raw text inspection workflow for this notebook section.'''
    text = read_text_file(path)
    lines = split_into_lines(text)
    summary = {
        'path': str(path),
        'total_lines': len(lines),
        'blank_lines': find_blank_lines(lines),
        'digit_lines': find_digit_lines(lines),
        'extra_space_lines': find_lines_with_extra_spaces(lines),
        'duplicate_lines': find_duplicate_lines(lines),
    }
    print_inspection_report(lines)
    return summary


inspection_summary = inspect_text_file()
inspection_summary

## Cleaning Text Lines

After inspection, the next task is normalization. This typically includes lowercasing text, trimming whitespace, removing punctuation, collapsing repeated spaces, and dropping lines that become empty after cleaning.

In [ ]:
def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Read a text file and return its lines without trailing newline characters.'''
    with path.open('r', encoding=encoding) as file:
        return file.read().splitlines()


def clean_line(line: str) -> str:
    '''Normalize a single line of text into a cleaner form.'''
    line = line.lower().strip()
    line = line.translate(str.maketrans('', '', string.punctuation))
    line = ' '.join(line.split())
    return line


def clean_lines(lines: list[str]) -> list[str]:
    '''Clean all lines and drop empty results.'''
    cleaned = []
    for line in lines:
        cleaned_line = clean_line(line)
        if cleaned_line != '':
            cleaned.append(cleaned_line)
    return cleaned


def write_lines(path: Path, lines: list[str], encoding: str = 'utf-8') -> None:
    '''Write lines to a UTF-8 text file, one line per output row.'''
    with path.open('w', encoding=encoding) as file:
        for line in lines:
            file.write(line + '\n')


def compare_raw_and_cleaned(raw_lines: list[str], cleaned_lines: list[str], limit: int = 5) -> None:
    '''Print a small side-by-side sample of raw and cleaned lines.'''
    print('Raw vs cleaned sample')
    for raw, cleaned in zip(raw_lines[:limit], cleaned_lines[:limit]):
        print(f'RAW   : {raw!r}')
        print(f'CLEAN : {cleaned!r}')
        print('---')


In [ ]:
def clean_text_lines(
    raw_path: Path = RAW_PATH,
    output_path: Path = GENERATED_CLEANED_PATH,
) -> list[str]:
    '''Run the line-cleaning workflow and save cleaned text.'''
    raw_lines = read_lines(raw_path)
    cleaned_lines = clean_lines(raw_lines)
    write_lines(output_path, cleaned_lines)

    print(f'Saved cleaned lines to: {output_path}')
    print(f'Raw line count: {len(raw_lines)}')
    print(f'Cleaned line count: {len(cleaned_lines)}')
    compare_raw_and_cleaned(raw_lines, cleaned_lines)
    return cleaned_lines


cleaned_lines_output = clean_text_lines()
cleaned_lines_output[:5]

## Building a Word Frequency Report

Once text lines are cleaned, they can be tokenized into words. A common analysis step is to remove stopwords, count how often each remaining word appears, sort the results, and save a frequency report that can be inspected later.

In [ ]:
def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Read a text file and return non-newline-stripped lines.'''
    with path.open('r', encoding=encoding) as file:
        return file.read().splitlines()


def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    '''Load stopwords from a text file into a set.'''
    lines = read_lines(path, encoding=encoding)
    return {line.strip() for line in lines if line.strip() != ''}


def tokenize_lines(lines: list[str]) -> list[str]:
    '''Split cleaned lines into a flat list of tokens.'''
    tokens = []
    for line in lines:
        tokens.extend(line.split())
    return tokens


def remove_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
    '''Remove stopwords from a token list.'''
    return [token for token in tokens if token not in stopwords]


def count_words(tokens: list[str]) -> dict[str, int]:
    '''Count token frequencies with a dictionary.'''
    counts: dict[str, int] = {}
    for token in tokens:
        counts[token] = counts.get(token, 0) + 1
    return counts


def sort_word_counts(word_counts: dict[str, int]) -> list[tuple[str, int]]:
    '''Sort word counts by descending frequency and then alphabetically.'''
    return sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))


def write_word_frequencies(
    path: Path,
    word_counts: list[tuple[str, int]],
    encoding: str = 'utf-8',
) -> None:
    '''Write word frequencies as tab-separated lines.'''
    with path.open('w', encoding=encoding) as file:
        for word, count in word_counts:
            file.write(f'{word}\t{count}\n')


In [ ]:
def build_word_frequency_report(
    cleaned_path: Path = GENERATED_CLEANED_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    output_path: Path = GENERATED_FREQ_PATH,
) -> list[tuple[str, int]]:
    '''Build and save a word frequency report from cleaned text.'''
    cleaned_lines = read_lines(cleaned_path)
    stopwords = load_stopwords(stopwords_path)
    tokens = tokenize_lines(cleaned_lines)
    filtered_tokens = remove_stopwords(tokens, stopwords)
    word_counts = count_words(filtered_tokens)
    sorted_counts = sort_word_counts(word_counts)
    write_word_frequencies(output_path, sorted_counts)

    print(f'Saved frequency report to: {output_path}')
    print(f'Token count: {len(tokens)}')
    print(f'Filtered token count: {len(filtered_tokens)}')
    print('Top 10 words:')
    for word, count in sorted_counts[:10]:
        print(f'{word}\t{count}')
    return sorted_counts


word_frequency_report = build_word_frequency_report()
word_frequency_report[:10]

## Refactoring with Comprehensions and Generators

The same workflow can be written in a more compact style by using comprehensions and generators. This section keeps the original steps, but rewrites parts of the cleaning and counting pipeline into shorter intermediate-Python patterns.

In [ ]:
def clean_line(line: str) -> str:
    '''Normalize one line of text.'''
    line = line.lower().strip()
    line = line.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(line.split())


def read_lines(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Read a UTF-8 text file into a list of lines.'''
    with path.open('r', encoding=encoding) as file:
        return file.read().splitlines()


def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    '''Load stopwords into a set.'''
    return {line.strip() for line in read_lines(path, encoding=encoding) if line.strip() != ''}


def cleaned_lines_from_file(path: Path, encoding: str = 'utf-8') -> list[str]:
    '''Read and clean lines using a list comprehension.'''
    lines = read_lines(path, encoding=encoding)
    return [clean_line(line) for line in lines if clean_line(line) != '']


In [ ]:
def cleaned_line_generator(path: Path, encoding: str = 'utf-8'):
    '''Yield cleaned non-empty lines one by one.'''
    for line in read_lines(path, encoding=encoding):
        cleaned = clean_line(line)
        if cleaned != '':
            yield cleaned


def token_generator(lines: list[str]):
    '''Yield tokens one by one from cleaned lines.'''
    for line in lines:
        for token in line.split():
            yield token


def build_word_counts(tokens: list[str], stopwords: set[str]) -> dict[str, int]:
    '''Build a frequency dictionary from filtered tokens.'''
    filtered_tokens = [token for token in tokens if token not in stopwords]
    unique_tokens = {token for token in filtered_tokens}
    return {token: filtered_tokens.count(token) for token in unique_tokens}


def summarize_counts(word_counts: dict[str, int], top_n: int = 10) -> list[tuple[str, int]]:
    '''Return the top N word counts.'''
    return sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))[:top_n]


def write_lines(path: Path, lines: list[str], encoding: str = 'utf-8') -> None:
    '''Write plain text lines to a file.'''
    with path.open('w', encoding=encoding) as file:
        for line in lines:
            file.write(line + '\n')


def write_word_frequencies(path: Path, pairs: list[tuple[str, int]], encoding: str = 'utf-8') -> None:
    '''Write word frequency pairs to a file.'''
    with path.open('w', encoding=encoding) as file:
        for word, count in pairs:
            file.write(f'{word}\t{count}\n')


In [ ]:
def run_comprehension_pipeline(
    raw_path: Path = RAW_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    cleaned_output_path: Path = GENERATED_CLEANED_V2_PATH,
    frequency_output_path: Path = GENERATED_FREQ_V2_PATH,
) -> dict[str, object]:
    '''Run the text pipeline in a more compact intermediate-Python style.'''
    stopwords = load_stopwords(stopwords_path)
    cleaned_lines = cleaned_lines_from_file(raw_path)
    tokens = list(token_generator(cleaned_lines))
    word_counts = build_word_counts(tokens, stopwords)
    top_words = summarize_counts(word_counts, top_n=10)

    write_lines(cleaned_output_path, cleaned_lines)
    write_word_frequencies(
        frequency_output_path,
        sorted(word_counts.items(), key=lambda item: (-item[1], item[0])),
    )

    print(f'Saved cleaned output to: {cleaned_output_path}')
    print(f'Saved frequency output to: {frequency_output_path}')
    print(f'Cleaned line count: {len(cleaned_lines)}')
    print(f'Token count: {len(tokens)}')
    print('Top 10 words:')
    for word, count in top_words:
        print(f'{word}\t{count}')

    return {
        'cleaned_lines': cleaned_lines,
        'tokens': tokens,
        'word_counts': word_counts,
        'top_words': top_words,
    }


comprehension_pipeline_results = run_comprehension_pipeline()
comprehension_pipeline_results['top_words']

## Packaging the Workflow in a Class

As the pipeline grows, it becomes useful to keep related data and behavior together. This section wraps the full workflow in a `TextCorpus` class that can load files, clean lines, tokenize text, calculate frequencies, save outputs, and summarize the results.

In [ ]:
def clean_line(line: str) -> str:
    '''Normalize one line of raw text.'''
    line = line.lower().strip()
    line = line.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(line.split())


class TextCorpus:
    '''A small reusable object for loading, cleaning, and analyzing text files.'''

    def __init__(self, raw_path: Path, stopwords_path: Path, encoding: str = 'utf-8'):
        self.raw_path = raw_path
        self.stopwords_path = stopwords_path
        self.encoding = encoding
        self.raw_lines: list[str] = []
        self.stopwords: set[str] = set()
        self.cleaned_lines: list[str] = []
        self.tokens: list[str] = []

    def load(self) -> None:
        '''Load raw lines and stopwords from disk.'''
        with self.raw_path.open('r', encoding=self.encoding) as file:
            self.raw_lines = file.read().splitlines()

        with self.stopwords_path.open('r', encoding=self.encoding) as file:
            self.stopwords = {line.strip() for line in file if line.strip() != ''}

    def clean(self) -> None:
        '''Clean raw lines and store the cleaned result.'''
        self.cleaned_lines = [clean_line(line) for line in self.raw_lines if clean_line(line) != '']

    def tokenize(self) -> None:
        '''Split cleaned lines into tokens.'''
        self.tokens = [token for line in self.cleaned_lines for token in line.split()]

    def filtered_tokens(self) -> list[str]:
        '''Return tokens with stopwords removed.'''
        return [token for token in self.tokens if token not in self.stopwords]

    def word_counts(self) -> dict[str, int]:
        '''Return token frequencies for filtered tokens.'''
        counts: dict[str, int] = {}
        for token in self.filtered_tokens():
            counts[token] = counts.get(token, 0) + 1
        return counts

    def top_words(self, n: int = 10) -> list[tuple[str, int]]:
        '''Return the top N words by frequency.'''
        counts = self.word_counts()
        return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:n]

    def save_cleaned(self, path: Path) -> None:
        '''Save cleaned lines to a text file.'''
        with path.open('w', encoding=self.encoding) as file:
            for line in self.cleaned_lines:
                file.write(line + '\n')

    def save_word_frequencies(self, path: Path) -> None:
        '''Save sorted word frequencies to a tab-separated text file.'''
        with path.open('w', encoding=self.encoding) as file:
            for word, count in sorted(self.word_counts().items(), key=lambda item: (-item[1], item[0])):
                file.write(f'{word}\t{count}\n')

    def summary(self) -> dict[str, int]:
        '''Return a compact numeric summary of the corpus.'''
        filtered = self.filtered_tokens()
        return {
            'raw_line_count': len(self.raw_lines),
            'cleaned_line_count': len(self.cleaned_lines),
            'token_count': len(self.tokens),
            'filtered_token_count': len(filtered),
            'unique_word_count': len(set(filtered)),
        }


In [ ]:
def run_text_corpus_pipeline(
    raw_path: Path = RAW_PATH,
    stopwords_path: Path = STOPWORDS_PATH,
    cleaned_output_path: Path = GENERATED_CLEANED_FINAL_PATH,
    frequency_output_path: Path = GENERATED_FREQ_FINAL_PATH,
) -> dict[str, object]:
    '''Run the full class-based text processing workflow.'''
    corpus = TextCorpus(raw_path, stopwords_path)
    corpus.load()
    corpus.clean()
    corpus.tokenize()
    corpus.save_cleaned(cleaned_output_path)
    corpus.save_word_frequencies(frequency_output_path)

    print(f'Saved cleaned output to: {cleaned_output_path}')
    print(f'Saved frequency output to: {frequency_output_path}')

    print('\nSummary')
    for key, value in corpus.summary().items():
        print(f'- {key}: {value}')

    print('\nTop 10 words')
    for word, count in corpus.top_words(10):
        print(f'{word}\t{count}')

    return {
        'summary': corpus.summary(),
        'top_words': corpus.top_words(10),
        'corpus': corpus,
    }


text_corpus_results = run_text_corpus_pipeline()
text_corpus_results['summary']

## Converting Survey Files to CSV

Text processing is also useful when the input is semi-structured rather than free-form. In this final section, a folder of survey text files is parsed with regular expressions, key values are extracted, longer answers are reduced to keywords, and the results are written into a CSV table.

In [ ]:
DEFAULT_OUTPUT_NAME = 'survey_responses_summary.csv'

QUESTION_FIELD_MAP = {
    1: 'name',
    2: 'age_answer',
    3: 'gender',
    4: 'contact_answer',
    5: 'employment_answer',
    6: 'education_answer',
    7: 'household_answer',
    8: 'income_answer',
    9: 'economic_pressures_answer',
    10: 'drive_answer',
    11: 'bike_answer',
    12: 'inflation_impact_answer',
    13: 'job_finance_security_answer',
    14: 'riga_economy_view_answer',
    15: 'improvements_answer',
}

TEXT_FIELDS_FOR_KEYWORDS = [
    'employment_answer',
    'education_answer',
    'household_answer',
    'income_answer',
    'economic_pressures_answer',
    'inflation_impact_answer',
    'job_finance_security_answer',
    'riga_economy_view_answer',
    'improvements_answer',
]

CSV_COLUMNS = [
    'source_file',
    'name',
    'age',
    'age_answer',
    'gender',
    'email',
    'phone',
    'employment_type',
    'occupation',
    'years_in_profession',
    'employment_answer',
    'employment_answer_keywords',
    'education_answer',
    'education_answer_keywords',
    'household_answer',
    'household_answer_keywords',
    'income_min_eur',
    'income_max_eur',
    'income_answer',
    'income_answer_keywords',
    'economic_pressures_answer',
    'economic_pressures_answer_keywords',
    'drive_km_week',
    'bike_km_week',
    'inflation_impact_answer',
    'inflation_impact_answer_keywords',
    'job_finance_security_answer',
    'job_finance_security_answer_keywords',
    'riga_economy_view_answer',
    'riga_economy_view_answer_keywords',
    'improvements_answer',
    'improvements_answer_keywords',
]


In [ ]:
def load_stopwords(path: Path, encoding: str = 'utf-8') -> set[str]:
    '''Load stopwords from a UTF-8 text file.'''
    with path.open('r', encoding=encoding) as file:
        return {line.strip().lower() for line in file if line.strip() != ''}


def read_text(path: Path, encoding: str = 'utf-8') -> str:
    '''Read a UTF-8 text file.'''
    with path.open('r', encoding=encoding) as file:
        return file.read()


def parse_survey_blocks(text: str) -> dict[int, str]:
    '''Extract numbered question-answer blocks with regex.'''
    pattern = re.compile(r'(?ms)^\s*(\d+)\.\s+.*?\n(.*?)(?=^\s*\d+\.\s+|\Z)')
    answers: dict[int, str] = {}

    for match in pattern.finditer(text):
        question_number = int(match.group(1))
        answer = match.group(2).strip()
        answers[question_number] = answer

    return answers


def normalize_text(text: str) -> str:
    '''Lowercase text and keep word-like tokens for keyword extraction.'''
    lowered = text.lower()
    cleaned = re.sub(r'[^\w\s]', ' ', lowered, flags=re.UNICODE)
    return ' '.join(cleaned.split())


def keyword_text(text: str, stopwords: set[str]) -> str:
    '''Remove stopwords from text and return normalized keywords.'''
    tokens = normalize_text(text).split()
    filtered_tokens = [token for token in tokens if token not in stopwords]
    return ' '.join(filtered_tokens)


def compact_value(text: str) -> str:
    '''Strip extra whitespace and trailing sentence punctuation.'''
    return text.strip().rstrip('.')


def first_int(text: str) -> int | None:
    '''Return the first integer found in text.'''
    match = re.search(r'\d+', text)
    return int(match.group()) if match else None


def income_range(text: str) -> tuple[int | None, int | None]:
    '''Extract minimum and maximum income values from Latvian survey text.'''
    numbers = [int(number) for number in re.findall(r'\d+', text)]
    if not numbers:
        return None, None
    if len(numbers) == 1:
        return numbers[0], numbers[0]
    return numbers[0], numbers[1]


def extract_email(text: str) -> str:
    '''Extract an email address from text if present.'''
    match = re.search(r'[\w.+-]+@[\w.-]+\.\w+', text, flags=re.UNICODE)
    return match.group(0) if match else ''


def extract_phone(text: str) -> str:
    '''Extract a Latvian-style phone number from text if present.'''
    match = re.search(r'\+371\s?\d{8}', text)
    return match.group(0) if match else ''


def employment_type(text: str) -> str:
    '''Return a compact employment type label from the employment answer.'''
    lowered = text.lower()
    if 'nepilnu slodzi' in lowered:
        return 'nepilna slodze'
    if 'pilnu slodzi' in lowered:
        return 'pilna slodze'
    if 'pensionār' in lowered:
        return 'pensionārs/pensionāre'
    return ''


def occupation(text: str) -> str:
    '''Extract the occupation phrase from the employment answer.'''
    match = re.search(r'\bpar (.+?)(?=,|\sun\b|$)', text, flags=re.IGNORECASE | re.UNICODE)
    if match:
        return compact_value(match.group(1))

    lowered = text.lower()
    if 'pensionāre' in lowered:
        return 'pensionāre'
    if 'pensionārs' in lowered:
        return 'pensionārs'
    return ''


In [ ]:
def survey_row(path: Path, stopwords: set[str]) -> dict[str, str | int | None]:
    '''Parse one survey text file into a flat CSV row.'''
    answers = parse_survey_blocks(read_text(path))
    parsed_answers = {
        field_name: answers.get(question_number, '')
        for question_number, field_name in QUESTION_FIELD_MAP.items()
    }

    row: dict[str, str | int | None] = {column: '' for column in CSV_COLUMNS}
    row['source_file'] = path.name

    row['name'] = compact_value(str(parsed_answers['name']))
    row['age_answer'] = str(parsed_answers['age_answer'])
    row['gender'] = compact_value(str(parsed_answers['gender']))
    row['employment_answer'] = str(parsed_answers['employment_answer'])
    row['education_answer'] = str(parsed_answers['education_answer'])
    row['household_answer'] = str(parsed_answers['household_answer'])
    row['income_answer'] = str(parsed_answers['income_answer'])
    row['economic_pressures_answer'] = str(parsed_answers['economic_pressures_answer'])
    row['inflation_impact_answer'] = str(parsed_answers['inflation_impact_answer'])
    row['job_finance_security_answer'] = str(parsed_answers['job_finance_security_answer'])
    row['riga_economy_view_answer'] = str(parsed_answers['riga_economy_view_answer'])
    row['improvements_answer'] = str(parsed_answers['improvements_answer'])

    row['age'] = first_int(str(parsed_answers['age_answer']))
    row['email'] = extract_email(str(parsed_answers['contact_answer']))
    row['phone'] = extract_phone(str(parsed_answers['contact_answer']))
    row['employment_type'] = employment_type(str(parsed_answers['employment_answer']))
    row['occupation'] = occupation(str(parsed_answers['employment_answer']))
    row['years_in_profession'] = first_int(str(parsed_answers['employment_answer']))
    income_min, income_max = income_range(str(parsed_answers['income_answer']))
    row['income_min_eur'] = income_min
    row['income_max_eur'] = income_max
    row['drive_km_week'] = first_int(str(parsed_answers['drive_answer']))
    row['bike_km_week'] = first_int(str(parsed_answers['bike_answer']))

    for field_name in TEXT_FIELDS_FOR_KEYWORDS:
        row[f'{field_name}_keywords'] = keyword_text(str(row[field_name]), stopwords)

    return row


def write_csv(
    path: Path,
    rows: list[dict[str, str | int | None]],
    encoding: str = 'utf-8',
) -> None:
    '''Write survey rows to CSV.'''
    with path.open('w', encoding=encoding, newline='') as file:
        writer = csv.DictWriter(file, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
def survey_folder_to_csv(
    input_folder: Path = SURVEYS_FOLDER,
    output_path: Path | None = None,
    stopwords_path: Path = STOPWORDS_PATH,
) -> list[dict[str, str | int | None]]:
    '''Run the survey folder to CSV pipeline with notebook-friendly defaults.'''
    input_folder = Path(input_folder)
    resolved_output_path = Path(output_path) if output_path else input_folder / DEFAULT_OUTPUT_NAME

    if not input_folder.exists() or not input_folder.is_dir():
        raise FileNotFoundError(f'Input folder not found: {input_folder}')

    stopwords = load_stopwords(stopwords_path)
    survey_paths = sorted(input_folder.glob('*.txt'))
    rows = [survey_row(path, stopwords) for path in survey_paths]

    write_csv(resolved_output_path, rows)

    print(f'Input folder: {input_folder}')
    print(f'Survey files processed: {len(rows)}')
    print(f'Saved CSV to: {resolved_output_path}')
    return rows


survey_rows = survey_folder_to_csv()
survey_rows[:2]